# **Project title:** Quantitative assessment of slc35a2 localization to the Golgi in single cells
### **Researcher:** Xavier Williams
### **Notebook author:** Shannon Rhoads (srhoads@unc.edu)

### **Editing & version log:**
- v0.1: 20260416 - downloaded [quality_check_segmentations.ipynb](https://github.com/SCohenLab/infer-subc/blob/main/notebooks/part_1_segmentation_workflows/quality_check_segmentations.ipynb) from [infer-subc version v2.0.0b1](https://github.com/SCohenLab/infer-subc/tree/v2.0.0b1)
- v0.2: 20260416 - the following edits were made:



----------
## **How to use Jupyter notebooks:**

Advance through each block of code below by pressing `Shift`+`Enter` or pressing the "Execute Cell" (`▶️`) button to the left of each block.

You will see a series of instructions before each block of code. Be on the look out for the following headers and follow the instructions accordingly:
- &#x1F3C3; **Run code; no user input required** - proceed without adding anything to the code block
- &#x1F453; **FYI** (for your information) - helpful information usually to bring context to what is going on
- &#x1F6D1; &#x270D; **User Input Required** - stop and input the appropriate information about your data. The following indicator will also be present in the code block:
   ```python 
   #### USER INPUT REQUIRED ###
   ```

----------

# **Notebook title:** Quality check segmentations

***Prior to this notebook, you should have completed at least one batch analysis on the Golgi and masks segmentations you wish to use for quantitative analysis.***

### **Purpose:**
1. **Recommended**: Visual inspection the segmentation outputs for each image in your dataset (***optional***)

    *We <ins>highly recommend</ins> going through this process. It is especially important for datasets where errors in the segmentations can have large impacts on the downstream statistical outcomes (e.g., datasets that do NOT have 100's of images per condition for example).*
2. **Recommended**: Application of automated quality checks to ensure the data is prepared for analysis in `Part 2 - Quantification`.
3. **REQUIRED**: Separation of the `masks` files into separate `cell` and `nuc`.

### 👣 **Summary of steps**  

➡️ **Input**
**The following files will be used as input in this notebook:**
1. "Raw" confocal microscopy image (".tiff", ".tif", or ".czi") where each channel of the image represents one of the organelles being segmented
2. The segmentation files created in the `batch_process_segmentation` notebook.

**You will also need the following:**
1. An empty folder to save segmentation files after editing
2. An empty folder to save the FINAL segmentation dataset (i.e., combining edited files with original segmentation files that did not need editing)

✅ **QUALITY CHECK SEGMENTATIONS**
- **`Step 1`** - Define the paths to your data and final output locations

- **`Step 2`** - Import and visualize segmentations

- **`Step 3`** - Edit individual segmentations (***optional***) 

- **`Step 4`** - Automated quality checks for segmentations before quantification (***recommended***)

- **`Step 5`** - Save organelle and region segmentations into specified folder

    - This step will also save the `cell` and `nuclei` as well as `soma` and `neurites` segmentations as separate files (***required***)

**Output** ➡️
**The output from this notebook will include:**
1. A folder containing any edited mask files saved as ".tiff"
2. A folder containing ALL FINAL segmentation files that are ready for quantification

---------------------
## **IMPORTS**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block imports the Python packages needed in the QC steps below.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import os
from pathlib import Path
import pandas as pd
import numpy as np

import napari
from napari.utils.colormaps import Colormap
from napari.settings import get_settings
settings = get_settings()
settings.application.ipy_interactive = True

viewer = napari.Viewer()

import colorsys
from infer_subc.core.file_io import (read_czi_image,
                                        export_inferred_organelle,
                                        import_inferred_organelle,
                                        list_image_files,)


import sys
sys.path.insert(0, str(Path("..").resolve()))
from src.segmentation import filter_segmentation, edit_segmentation

from bioio import BioImage

%load_ext autoreload
%autoreload 2

c:\Users\Shannon\anaconda3\envs\golgi-coloc\lib\site-packages\npe2\manifest\_npe1_adapter.py:87: UserWarning: Error importing contributions for first-generation napari plugin 'napari-aicsimageio': cannot import name 'Dimensions' from 'aicsimageio.constants' (c:\Users\Shannon\anaconda3\envs\golgi-coloc\lib\site-packages\aicsimageio\constants.py)
  self._load_contributions()


-------
### **`STEP 1` - Define the paths to your data**

#### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** This code block specified all of the information needed to import your raw and segmentation data. The information is collected in list. The orders of these lists are matched so that the first item in one list corresponds to the first item in the other list. 

Please indicate whether you will use sample data or specify the following information about your data: 
- `raw_path`: A string of the path to your raw (e.g., intensity) images that were used as the input for segmentation
- `seg_path`: A string of the path where the segmentation outputs are saved
- `location_tosave_fullset_gooddata`: A string to an empty folder that will contain all of the FINAL segmentation files (e.g., it will combine the newly saved edited segmentation files with any of the original segmentation files that did not need editing)
- `raw_file_type`: The raw file type (e.g., ".tiff" or ".czi")
- `seg_file_type`: The segmentation file type (e.g., ".tiff" or ".tif" if you were using infer-subc notebooks or Napari plugin)
- `suffixes`: A list of the endings of your segmentation file names; this usually just includes the segmentation suffix (e.g., 'lyso'), but may also include additional text if the settings file in Napari or the 'name_suffix' in the batch_process_segmentation notebook contained additional text.
- `channels`: A list of indexes of the channels in the raw intensity image used to generate the segmentation corresponding images. The `channels` order should match the order of the `suffixes`. Channel indexes begin at 0 for channel 1. For objects that have no specific channels, please input a value of None.*
- `multichannel_names`: A dictionary with values in suffixes assigned to a list of names to split the multichannel segmentation into. These values are only the suffix values that correspond to images with multiple channels. For example, {'mask': ['nuc', 'cell']} due to the 'mask' file containing both the cell mask and nucleus segmentations.*

In [2]:
#### USER INPUT REQUIRED ###
raw_data = Path(os.path.dirname(os.getcwd()))  / "pilot-data"
seg_path = Path(os.path.dirname(os.getcwd()))  / "test-segmentation"

location_tosave_edited_segmentations = Path(os.path.dirname(os.getcwd()))  / "test-segmentation/edited"
location_tosave_fullset_gooddata = Path(os.path.dirname(os.getcwd()))  / "test-segmentation/good_segs"

raw_file_type = ".czi"
seg_file_type = ".tiff"

suffixes = ['golgi']
channels = [0]
multichannel_names = None

#### &#x1F6D1; &#x270D; ***Optional* User Input Required:**

&#x1F453; **FYI:** If you ran the batch processing more than once, you can include information about the additional segmentation location and file suffixes here. If you do **NOT** have a second file location, for each object below, please indicate: `None`

***Important:** Only one location for each segmentation type will be used. If `lyso` and `lyso_2` are both specified at the same location in the list, **ONLY** `lyso_2` will be visualized in the below steps.*

In [3]:
#### Optional - USER INPUT REQUIRED ###
seg_path_2 = None

suffixes_2 = [None]

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block will print the list of raw file names in your raw_path. You will want to review each file in the list for accurate segmentations. The next set of code blocks will walk you how to do this.

In [4]:
raw_file_list = list_image_files(Path(raw_data),raw_file_type)

pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":raw_file_list})

,Image Name
0,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026A88PB1T1P1-01.czi
1,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026GLN108STOPB1T1P1-08.czi
2,c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\03022026R55LB1T1P1-01.czi


### **`STEP 2` - Import and visualize organelle and region segmentations**

#### &#x1F6D1; &#x270D; **User Input Required:**
&#x1F53C; Use the list  above to determine the index of the image you would like to look at. Eventually, you should look at all images. The pipeline below will indicate when you need to return here to repeat for the next image in the list.

In [5]:
#### USER INPUT REQUIRED ###
num = 0

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block will import the segmentation files associated to the raw image you selected above. The raw image and segmentations will be included in a Napari viewer for visual inspection.

In [6]:
file_path = str(raw_file_list[num])
raw_file = BioImage(file_path)
raw_img_data = np.squeeze(raw_file.data)
raw_meta_dict = {'file_name': file_path}

viewer.layers.clear()
viewer.add_image(raw_img_data, name='raw', blending='additive')
print("Image name:")
print(raw_file_list[num].name)
obj_segs = {}

# Importing segmentations
for suffix, channel in zip(suffixes, channels):
    obj = import_inferred_organelle(suffix, raw_meta_dict, Path(seg_path), seg_file_type)
    if multichannel_names is None:
        multichannel_names = {}
    if suffix not in multichannel_names.keys():
        if len(obj.shape) > 3:
            print(f"{suffix} is multichannel and not in multichannel_names variable, splitting into unnamed separate channels")
            for i in range(int(obj.shape[0])):
                obj_segs[f"{suffix}_{i}"] = (obj[i], channel)
        else:
            obj_segs[suffix] = (obj, channel)
    else:
        if len(obj.shape) <= 3:
            print(f"{suffix} is not multichannel, ignoring")
            obj_segs[suffix] = (obj, channel)
        else:
            for i in range(int(obj.shape[0])):
                obj_segs[multichannel_names[suffix][i]] = (obj[i], channel)

for suffix2, suffix, channel in zip(suffixes_2, suffixes, channels):
    if suffix2 is not None:
        obj = import_inferred_organelle(suffix2, raw_meta_dict, Path(seg_path_2), seg_file_type)
        if suffix not in multichannel_names.keys():
            if len(obj.shape) > 3:
                print(f"{suffix2} is multichannel and not in multichannel_names variable, splitting into unnamed separate channels")
                for i in range(int(obj.shape[0])):
                    obj_segs[f"{suffix}_{i}"] = (obj[i], channel)
            else:
                obj_segs[suffix] = (obj, channel)
        else:
            if len(obj.shape) <= 3:
                print(f"{suffix2} is not multichannel, ignoring")
                obj_segs[suffix] = (obj, channel)
            else:
                for i in range(int(obj.shape[0])):
                    obj_segs[multichannel_names[suffix][i]] = (obj[i], channel)

# Creating colorways
HSV_tuples = [(i/(len(obj_segs.keys())), 1, 1) for i in range(len(obj_segs.keys()))]
RGB_tuples = map(lambda x: colorsys.hsv_to_rgb(*x), HSV_tuples)
col = ["".join("%02X" % round(c*255) for c in rgb) for rgb in RGB_tuples]

for count, suffix in enumerate(obj_segs.keys()):
    cmap = Colormap(["#000000", f"#{col[count]}"], 
                    name=f'{suffix}_cmap')
    channel = obj_segs[suffix][1]
    obj = obj_segs[suffix][0]
    
    if channel is not None:
        viewer.add_image(raw_img_data[channel], name=f'{suffix}_raw', blending='additive')
    viewer.add_image(obj, opacity=0.3, name=f'{suffix}_seg', colormap=cmap, contrast_limits=[0,1])
obj_segs_edit = {key: None for key in obj_segs.keys()}

Image name:
03022026A88PB1T1P1-01.czi
loaded  inferred 3D `golgi`  from c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\test-segmentation 


### **`STEP 3` - Edit segmentations**

Now that you visually inspected your image and the segmentation output files, indicate below which of the segmentations you would like to edit, if any.

This step may be skipped if you are already satisfied with your segmentations for this image.

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** The below block of code lists out the segmentations that you have viewed. 

In [7]:
print(list(obj_segs.keys()))

['golgi']


#### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F53C; Use the above list, type one of the following options for each segmentation type corresponding to the segmentations listed by the above block of code:
- `True`: for any images you want to edit
- `False`: for any that you do not want to edit


In [8]:
obj_edit = [True]

#### &#x1F3C3; **Run code; no user input required**

For any image that you have decided to edit (specified True above), a new Napari viewer will opened one at a time. When you are finished editing the image, close the Napari viewer (do not minimize it), and the code will proceed to the next image. This will repeat until you have cycled through all of your images. If you ever wish to restart editing any of the segmentations from the beginning, you can rerun this block of code after setting the segmentations you do not wish to edit again as false in the obj_edit variable above and rerunning that block of code. 

If you wish to create a segmentation from scratch, you must still edit the layer that you are told to as this is the only layer that will be saved. To start from scratch, you will need to erase any segmentation already present in the image.

Once all segmentaitons are edited, close all Napari windows to finish execution of this code block.

In [9]:
if len(obj_edit) != len(obj_segs.keys()):
    print("The number of values in the obj_edit variable is not equal to the number of object segmentations.")
    print("Please adjust the values in the obj_edit variable above, then rerun this block of code.")
elif not any(obj_edit):
    print("You have selected to not edit any segmentations.")
elif any(obj_edit):
    # select the keys of the segmentations to be edited based on the obj_edit variable and print them here
    edit_keys = [suffix for edit, suffix in zip(obj_edit, obj_segs.keys()) if edit]
    print(f"You have selected to edit the following segmentations: {edit_keys}")

for edit, suffix in zip(obj_edit, obj_segs.keys()):
    if (obj_segs_edit[suffix] is None) or edit:
        obj_segs_edit[suffix] = edit_segmentation(suffix, viewer, edit)
        if edit:
            print(f"Saving your edited {suffix} segmentation to disk...")
            export_inferred_organelle(obj_segs_edit[suffix], suffix, raw_meta_dict, Path(location_tosave_edited_segmentations))

print("If you would like to go back and edit any of these segmentations, you can change the obj_edit variable to only select for objects desired to be edited and rerun this block of code.")

You have selected to edit the following segmentations: ['golgi']

A new Napari viewer has been opened.
You may now begin editing the golgi segmentation.
Head to the Napari window to edit your golgi segmentation output in the golgi_seg layer.
When you close out of the viewer, the edited golgi segmentation will be saved automatically
Saving your edited golgi segmentation to disk...
saved file: 03022026A88PB1T1P1-01-golgi
If you would like to go back and edit any of these segmentations, you can change the obj_edit variable to only select for objects desired to be edited and rerun this block of code.


> #### &#x1F6D1; **STOP: Use the `Napari` window to edit chosen segmentations.** &#x1F50E;
>
> **At this point, a `Napari` window will have opened one of the chosen segmentations to edit. Within this Napari window, do NOT change the names of any of the layers. Once you have finished editing the image, do the following:**
> 1) You may manually save the segmentation to another location; however, this occurs automatically and will overwrite your saved segmentations. (optional)
> 2) Close out of all `Napari` windows (do not minimize).
>
> A new `Napari` window will open up so long as there are more segmentations you have chosen to edit.



> #### **Once **`ALL`** segmentaitons are edited, close **`ALL`** Napari windows to finish execution of this code block.**

### **`STEP 4` - Save segmentations**

#### &#x1F3C3; **Run code; no user input required**

Now that your segmentations have passed quality control, you can save them using the below block of code.

In [13]:
for suffix in obj_segs.keys():
    try:
        obj_segs_edit[suffix] = import_inferred_organelle(suffix, raw_meta_dict, Path(location_tosave_edited_segmentations), seg_file_type)
    except (FileNotFoundError):
        obj_segs_edit[suffix] = obj_segs[suffix][0]
        
for suffix in obj_segs_edit.keys():
    print(f"Saving your final {suffix} segmentation to disk...")
    export_inferred_organelle(obj_segs_edit[suffix], suffix, raw_meta_dict, Path(location_tosave_fullset_gooddata))

loaded  inferred 3D `golgi`  from c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\test-segmentation\edited 
Saving your final golgi segmentation to disk...
making c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\test-segmentation\good_segs
saved file: 03022026A88PB1T1P1-01-golgi


> ### 🔁🔁 **Repeat `Steps 2-4` above for *EACH IMAGE* in your dataset.**  
>
> You've now completed the quality check for one image. Repeat these steps for each image in your dataset by selecting the next image in the list at the beginning of `Step 2` above.

-----
### 🎉 **CONGRATULATIONS!! You've successfully completed `infer-subc Part 1 - Segmentation Workflows`.**

Continue on to quantitative analysis in the [] notebook.